# Fast Fourier Transform (FFT)

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

The Fast Fourier Transform converts the signal from the time domain to the frequency domain, revealing the energy carried by the signal at each frequency. We shade the five brain wave bands (delta, theta, alpha, beta, gamma) to link the spectrum to physiological meanings.

## Expected outputs

- A prominent peak in the alpha band (8-13 Hz) reflecting visual cortex activity
- Energy in delta and theta bands common in raw EEG data
- Gradual decrease in energy with increasing frequency (1/f law)

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| Sampling rate | 200 Hz | One sample every 5 ms |
| Frequencies | 0-80 Hz | Analysis range |
| Bands | 5 | delta, theta, alpha, beta, gamma |


## 1. Install dependencies


In [ ]:
!pip install scipy numpy plotly wfdb pywt


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2, channel P4 (parietal region).


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


## 4. Apply the FFT

We apply `scipy.fft.fft` to the full signal, then extract the absolute value to compute the frequency magnitude. We keep only positive frequencies since the negative half mirrors the positive half.


In [ ]:
from scipy.fft import fft, fftfreq

spectrum = fft(channel_data)
freqs = fftfreq(len(channel_data), 1 / fs)
magnitude = np.abs(spectrum)
pos_mask = freqs >= 0
freqs = freqs[pos_mask]
magnitude = magnitude[pos_mask]
print(f'Frequency range: {freqs.min():.1f} - {freqs.max():.1f} Hz')


## 5. Interactive plot

**What to look for:**

- Prominent peaks indicate the most present frequencies in the signal
- Colored shading marks the five brain wave bands
- Use the zoom tool to inspect specific frequency ranges



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = np.arange(n_plot) / fs

BANDS = [
    ('Delta', 0.5, 4, 'green'),
    ('Theta', 4, 8, 'blue'),
    ('Alpha', 8, 13, 'orange'),
    ('Beta', 13, 30, 'red'),
    ('Gamma', 30, 80, 'purple'),
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=('Original signal - Channel P4',
                                    'FFT Spectrum - Channel P4'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Signal',
                         line=dict(color='gray', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=freqs, y=magnitude, name='Magnitude',
                         line=dict(color='black', width=0.8)), row=2, col=1)
for name, fmin, fmax, color in BANDS:
    fig.add_vrect(x0=fmin, x1=fmax, fillcolor=color, opacity=0.1,
                  line_width=0, row=2, col=1)
fig.update_xaxes(range=[0, 80], row=2, col=1)
fig.update_layout(height=700, title_text='FFT Analysis - Channel P4',
                  xaxis_title='Time (s)', xaxis2_title='Frequency (Hz)',
                  yaxis_title='Amplitude (uV)', yaxis2_title='Magnitude',
                  showlegend=False)
fig.show()


## What did we learn?

- Fourier transform reveals the stationary frequency components in the signal
- Spectrum peaks correspond to the most present frequencies
- The transform gives an average spectrum over the entire duration and does not tell when each frequency appeared
- The wavelet transform addresses this limitation with a time-frequency analysis

